# Exploratory analysis: is the motor-imagery signal there?

One subject at a time, with the same package functions the figure script uses. The final group figures come from `scripts/make_eda_figures.py`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.signal import welch

from sixthfinger.config import load_data_config
from sixthfinger.data.epochs import load_windows
from sixthfinger.paths import get_paths
from sixthfinger.viz import topo
from sixthfinger.viz.plots import CLASS_COLORS, CLASS_SHORT, use_style

use_style()
cfg, paths = load_data_config(), get_paths()
SUBJECT, SESSION = "sub-01", "pre"
c3 = cfg.channels.keep.index("C3")

## Spectra of the cached 4 s windows at C3

Imagery should lower the 8–13 Hz (mu) power compared with rest.

In [ ]:
X, meta = load_windows(paths.epochs, cfg, subjects=[SUBJECT], sessions=[SESSION], mirror=True)
f, p = welch(X[:, c3], fs=250.0, nperseg=500, axis=-1)
fig, ax = plt.subplots(figsize=(4, 2.6))
for name, color in CLASS_COLORS.items():
    sel = (meta.label_name == name).to_numpy()
    ax.semilogy(f, p[sel].mean(axis=0), color=color, label=CLASS_SHORT[name])
ax.set_xlim(2, 40)
ax.axvspan(8, 13, color="0.9", zorder=0)
ax.set_xlabel("Hz")
ax.legend(frameon=False);

## ERD around the cue

Windows from 1 s before to 5 s after the imagery cue, cut on the fly from the processed runs. ERD is the change in power relative to the second before the cue: negative means the rhythm got weaker.

In [ ]:
Xl, y = topo.cue_windows(SUBJECT, SESSION, cfg, paths.processed)
fig, axes = plt.subplots(1, 2, figsize=(7, 2.6), sharey=True)
for ax, (label, name) in zip(axes, [(1, "affected_hand"), (2, "sixth_finger")], strict=True):
    erd, times = topo.erd_timefreq(Xl[y == label])
    ax.pcolormesh(times, topo.TFR_FREQS, erd[c3], cmap="RdBu_r", vmin=-60, vmax=60, shading="auto")
    ax.axvline(0, color="k", lw=0.8)
    mu = topo.band_mean(erd, times, topo.BANDS["mu"])[c3]
    ax.set_title(f"{CLASS_SHORT[name]}: mu ERD at C3 {mu:.1f} %")
    ax.set_xlabel("time from cue (s)")
axes[0].set_ylabel("Hz");

## Where on the head

Mean mu ERD over 1–5 s after the cue, per electrode.

In [ ]:
info = topo.montage_info(cfg.channels.keep)
fig, axes = plt.subplots(1, 2, figsize=(5, 2.4))
for ax, (label, name) in zip(axes, [(1, "AH"), (2, "SF")], strict=True):
    erd, times = topo.erd_timefreq(Xl[y == label])
    topo.band_topomap(topo.band_mean(erd, times, topo.BANDS["mu"]), info, ax, name)